# 02 - Verificacion de Eventos en Kafka

Este notebook verifica que los eventos del producer esten llegando correctamente a Kafka.

## 1. Instalacion de kafka-python

In [1]:
import sys
!{sys.executable} -m pip install kafka-python-ng pandas python-dotenv

## 2. Consumo de mensajes de Kafka

In [2]:
from kafka import KafkaConsumer
import json

KAFKA_BROKER = "kafka:29092"
TOPIC = "iot.air_quality.streaming"

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset="earliest",
    consumer_timeout_ms=10000,
    value_deserializer=lambda m: json.loads(m.decode("utf-8"))
)

print(f"Consumiendo mensajes del topico: {TOPIC}")
print("Esperando 10 segundos...\n")

messages = []
for msg in consumer:
    messages.append(msg.value)
    print(f"[{len(messages)}] Estacion: {msg.value.get('estacion')} | Temp: {msg.value.get('temperatura')} | IAQ: {msg.value.get('iaq')}")

print(f"\nTotal mensajes recibidos: {len(messages)}")
consumer.close()

Consumiendo mensajes del topico: iot.air_quality.streaming
Esperando 10 segundos...

[1] Estacion: ESP32_01 | Temp: 25.4 | IAQ: 39
[2] Estacion: ESP32_02 | Temp: 25.14 | IAQ: 45
[3] Estacion: ESP32_03 | Temp: 24.79 | IAQ: 44
[4] Estacion: ESP32_04 | Temp: 25.53 | IAQ: 65
[5] Estacion: ESP32_05 | Temp: 25.36 | IAQ: 42
[6] Estacion: ESP32_01 | Temp: 25.4 | IAQ: 40
[7] Estacion: ESP32_02 | Temp: 25.25 | IAQ: 37
[8] Estacion: ESP32_03 | Temp: 24.87 | IAQ: 38
[9] Estacion: ESP32_04 | Temp: 25.23 | IAQ: 59
[10] Estacion: ESP32_05 | Temp: 25.43 | IAQ: 47
[11] Estacion: ESP32_01 | Temp: 25.4 | IAQ: 41
[12] Estacion: ESP32_02 | Temp: 25.13 | IAQ: 40
[13] Estacion: ESP32_03 | Temp: 25.24 | IAQ: 44
[14] Estacion: ESP32_04 | Temp: 25.12 | IAQ: 65
[15] Estacion: ESP32_05 | Temp: 25.38 | IAQ: 45
[16] Estacion: ESP32_01 | Temp: 25.4 | IAQ: 47
[17] Estacion: ESP32_02 | Temp: 25.33 | IAQ: 41
[18] Estacion: ESP32_03 | Temp: 26.17 | IAQ: 39
[19] Estacion: ESP32_04 | Temp: 25.22 | IAQ: 57
[20] Estacion: E

## 3. Analisis de mensajes recibidos

In [3]:
import pandas as pd

if messages:
    df = pd.DataFrame(messages)
    print("\nEstructura del evento:")
    print(df.columns.tolist())
    print("\nPrimer evento completo:")
    print(df.iloc[0].to_dict())
    print("\nEstaciones detectadas:")
    print(df["estacion"].value_counts())
else:
    print("No se recibieron mensajes. Verifica que el producer este corriendo.")


Estructura del evento:
['estacion', 'temperatura', 'humedad', 'presion', 'altura', 'gas', 'iaq', 'eco2', 'VOC', 'calidad_aire', 'created_at', 'event_timestamp', 'delayed']

Primer evento completo:
{'estacion': 'ESP32_01', 'temperatura': 25.4, 'humedad': 58.0, 'presion': 1013.45, 'altura': 3820, 'gas': 117, 'iaq': 39, 'eco2': 601, 'VOC': 0.487, 'calidad_aire': 'BUENA', 'created_at': '2026-05-19T15:05:20.838540', 'event_timestamp': 1779203120838, 'delayed': nan}

Estaciones detectadas:
estacion
ESP32_01    5293
ESP32_02    5293
ESP32_03    5293
ESP32_04    5293
ESP32_05    5293
Name: count, dtype: int64


## 4. Verificacion de eventos retrasados (ESP32_05)

In [4]:
if messages:
    delayed = [m for m in messages if m.get("delayed", False)]
    print(f"Eventos retrasados detectados: {len(delayed)}")
    if delayed:
        print("\nEjemplo de evento retrasado:")
        print(f"  created_at: {delayed[0].get('created_at')}")
        print(f"  estacion: {delayed[0].get('estacion')}")
        print(f"  delayed: {delayed[0].get('delayed')}")

Eventos retrasados detectados: 5293

Ejemplo de evento retrasado:
  created_at: 2026-05-19T15:05:16.129393
  estacion: ESP32_05
  delayed: True


## 5. Verificacion de latencia

In [5]:
from datetime import datetime

if messages:
    latencias = []
    now = datetime.utcnow().timestamp() * 1000
    for m in messages:
        if m.get("event_timestamp"):
            lat = now - m["event_timestamp"]
            latencias.append(lat)
    
    if latencias:
        print(f"Latencia promedio: {sum(latencias)/len(latencias):.0f} ms")
        print(f"Latencia minima: {min(latencias):.0f} ms")
        print(f"Latencia maxima: {max(latencias):.0f} ms")

Latencia promedio: 375449268 ms
Latencia minima: 10394 ms
Latencia maxima: 440641660 ms
